In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import os
import cmcrameri.cm as cmc
import warnings
from matplotlib.colors import LogNorm
warnings.filterwarnings("ignore")

project_root = r'c:\Users\Timothee Delcourt\Documents\ETHZ-PhD\Coding\mars_currents_pinn'
os.chdir(project_root)
print("Project root directory:", os.getcwd())


Project root directory: c:\Users\Timothee Delcourt\Documents\ETHZ-PhD\Coding\mars_currents_pinn


In [ ]:
cmc.show_cmaps()

### XZ plane

In [3]:
# set up
def plot_data_mso(x_axis = 'X', y_axis = 'Z'):
    Rm = 3393.5 # km
    plt.figure(figsize=(8,6))
    x_planet = np.arange(-1,1+0.0001,0.0001)
    y_planet = np.sqrt(1-x_planet**2)
    plt.plot(x_planet,y_planet,color='k')
    plt.plot(x_planet,-y_planet,color='k')
    
    plt.axis('equal')
    plt.xlabel(x_axis+r'$_{\text{MSO}}$')
    plt.ylabel(y_axis+r'$_{\text{MSO}}$')
    plt.xlim([-3,3])
    plt.ylim([-3,3])

    # plot data
    detla_km = 200
    input = torch.load('data/position_mso.pt')
    crustal_field_mso = torch.load('data/crustal_field_mso.pt')
    observation_mso = torch.load('data/observation_mso.pt')
    target = observation_mso - crustal_field_mso
    if (x_axis != 'Y') & (y_axis != 'Y'):
        plt.fill_between(x_planet, -y_planet, y_planet, where=(x_planet<=0), color='k')
        condition = (input[:,1] <= detla_km) & (input[:,1] >= -detla_km)
        i_x, i_y = 0, 2
    elif (x_axis != 'X') & (y_axis != 'X'):
        condition = (input[:,0] <= detla_km) & (input[:,0] >= -detla_km)
        i_x, i_y = 1, 2
    elif (x_axis != 'Z') & (y_axis != 'Z'):
        plt.fill_between(x_planet, -y_planet, y_planet, where=(x_planet<=0), color='k')
        condition = (input[:,2] <= detla_km) & (input[:,2] >= -detla_km)
        i_x, i_y = 0, 1
    input = input[condition]
    input/=Rm # km -> Rm
    target = target[condition]
    x, z = input[:,i_x].numpy(), input[:,i_y].numpy()
    B_total = torch.sqrt(target[:,0]**2 + target[:,1]**2 + target[:,2]**2).numpy()
    plt.scatter(x, z, c=B_total, cmap=cmc.batlow, s=1, norm=LogNorm())
    plt.colorbar(label='B (nT)')
    plt.tight_layout()
    plt.savefig('figures/data_MSO/MSO_'+x_axis+y_axis+'png',dpi=300)
    plt.close()

# plot_data_mso()
plot_data_mso('X','Y')
plot_data_mso('Y','Z')